## Proof of concept: prompt repitition with Mistral 7B via MLX
**Goal**: validate the pipeline on 20 ARC-Challenge questions before committing to a full pilot run
* **Hardware**: M2 Macbook Air, 16GB unified memory
* **Model**: Mistral-7B-Instruct
* **Expected Runtime**: 5-10 minutes for 20 question with 2 variants
---

### Validation
1. Model loads and generates coherent answers
2. `PromptBuilder` produces correct baseline and repeat variants
3. `Evalutator` parses Mistral's output reliably
4. Paired results are structued correctly for McNemar testing later
5. Latency and memory usage are acceptable for M2

#### Depencies

In [1]:
import sys, os, json, time, logging
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
print("Python", sys.version.split()[0])

Python 3.11.0


#### Load ARC-Challenge

In [2]:
from data.arc_loader import ARCLoader

loader = ARCLoader(split='test')
examples = loader.load_subset(n=100, seed=42)

print(f"Loaded {len(examples)} ARC-Challenge questions")
print("\nSample question:")
ex = examples[0]
print(f" ID     : {ex.example_id}")
print(f" Q      : {ex.question[:80]}...")
print(f" Options: {ex.options}")
print(f" Gold   : {ex.correct_label}")

/Users/lboyer/Documents/Python/prompt_repitition_replication/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/ai2_arc' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR  `trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/ai2_arc' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
INFO  HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/re

Loaded 100 ARC-Challenge questions

Sample question:
 ID     : Mercury_SC_407400
 Q      : A class plans an investigation to see which brand of light bulb lasts the longes...
 Options: ['Repeat the investigation.', 'Write a report of the results.', 'Make a table for recording data.', 'Make daily observations of the light bulbs.']
 Gold   : C


#### Build Prompt Variants

In [3]:
from agents.prompt_builder import PromptBuilder, PromptVariant

builder = PromptBuilder()

def make_prompt(example, variant: PromptVariant) -> str:
    options_text = "\n".join(
        f"{letter}) {text}" for letter, text in zip("ABCD", example.options)
    )
    query = (
        f"{example.question}\n\n"
        f"{options_text}\n\n"
        "Provide your answer in the format: The answer is X "
        "(where X is A, B, C, or D)."
    )
    return builder.build(query, variant)

# sanity check
sample_base = make_prompt(examples[0], PromptVariant.BASELINE)
sample_rep = make_prompt(examples[0], PromptVariant.REPEAT)

base_query = sample_base.split("\n\n", 1)[1]
rep_query = sample_rep.split("\n\n", 1)[1]

assert len(rep_query) == 2 * len(base_query), "Repeat is not exact x2 baseline"
print(" Prompt lengths verified.")
print(f"  Baseline query chars  :  {len(base_query)}")
print(f"  Repeat query chars  :  {len(rep_query)}")

 Prompt lengths verified.
  Baseline query chars  :  349
  Repeat query chars  :  698


#### Load MLX Model

In [4]:
# Memory budget on 16 GB M2 Air:
#   macOS headroom  : ~3–4 GB
#   Mistral 7B 4-bit: ~4.5 GB
#   Python/notebook : ~0.5 GB
#   ─────────────────────────
#   Remaining buffer: ~7 GB        comfortable i guess

from agents.mlx_agent import MLXAgent, MLXAgentConfig

config = MLXAgentConfig(
    max_tokens=256,             # default 128 but there was one discordant result that was clearly a cap artifact
    temperature=0.0,
    verbose=False,
)

print("Loading model (first run downloads weights)...")
t0 = time.perf_counter()
agent = MLXAgent(config)
print(f"Model loaded in {time.perf_counter() - t0:.1f} s")

Loading model (first run downloads weights)...


INFO  Loading MLX model: mlx-community/Mistral-7B-Instruct-v0.3-4bit …
INFO  HTTP Request: GET https://huggingface.co/api/models/mlx-community/Mistral-7B-Instruct-v0.3-4bit/revision/main "HTTP/1.1 200 OK"
Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 86100.08it/s]
INFO  Model loaded in 2.7 s.


Model loaded in 4.1 s


#### Manual Sanity Check on Q1

In [5]:
test_ex = examples[0]

print("=== BASELINE ===")
r_base = agent.query(make_prompt(test_ex, PromptVariant.BASELINE), variant="baseline")
print(r_base.text)
print(f"Latency: {r_base.latency_ms:.0f} ms | tokens: {r_base.completion_tokens}")

print("\n=== REPEAT ===")
r_rep = agent.query(make_prompt(test_ex, PromptVariant.REPEAT), variant="repeat")
print(r_rep.text)
print(f"Latency: {r_rep.latency_ms:.0f} ms | tokens: {r_rep.completion_tokens}")

print(f"\nGold answer: {test_ex.correct_label}")

=== BASELINE ===
The answer is D) Make daily observations of the light bulbs.
Latency: 2162 ms | tokens: 15

=== REPEAT ===
The answer is D) Make daily observations of the light bulbs.
Latency: 2300 ms | tokens: 15

Gold answer: C


## Run full 100 Question Paired Pilot

In [6]:
from experiments.evaluator import Evaluator

evaluator = Evaluator()
records = []

for i, ex in enumerate(examples):
    # both prompts
    p_base = make_prompt(ex, PromptVariant.BASELINE)
    p_rep  = make_prompt(ex, PromptVariant.REPEAT)

    # query model (same question, both variants)
    r_base = agent.query(p_base, variant="baseline")
    r_rep  = agent.query(p_rep,  variant="repeat")

    # evaluate
    ev_base = evaluator.evaluate(r_base.text, ex.correct_label, question_id=ex.example_id, variant="baseline")
    ev_rep  = evaluator.evaluate(r_rep.text,  ex.correct_label, question_id=ex.example_id, variant="repeat")

    records.append({
        "question_id"        : ex.example_id,
        "gold"               : ex.correct_label,
        # baseline
        "base_correct"       : ev_base.is_correct,
        "base_predicted"     : ev_base.predicted,
        "base_parse_status"  : ev_base.parse_status.value,
        "base_latency_ms"    : r_base.latency_ms,
        "base_tokens"        : r_base.completion_tokens,
        # repeat
        "rep_correct"        : ev_rep.is_correct,
        "rep_predicted"      : ev_rep.predicted,
        "rep_parse_status"   : ev_rep.parse_status.value,
        "rep_latency_ms"     : r_rep.latency_ms,
        "rep_tokens"         : r_rep.completion_tokens,
    })

    # progress dot every 5 questions
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{len(examples)} done")

print("\nPilot run complete.")

  5/100 done
  10/100 done
  15/100 done
  20/100 done
  25/100 done
  30/100 done
  35/100 done
  40/100 done
  45/100 done
  50/100 done
  55/100 done
  60/100 done
  65/100 done
  70/100 done
  75/100 done
  80/100 done
  85/100 done
  90/100 done
  95/100 done
  100/100 done

Pilot run complete.


## Results

In [7]:
df = pd.DataFrame(records)

n = len(df)
base_acc = df["base_correct"].mean()
rep_acc  = df["rep_correct"].mean()

print(f"Questions     : {n}")
print(f"Baseline acc  : {base_acc:.1%}  ({df['base_correct'].sum()}/{n})")
print(f"Repeat acc    : {rep_acc:.1%}  ({df['rep_correct'].sum()}/{n})")
print(f"Δ accuracy    : {rep_acc - base_acc:+.1%}")
print()

# McNemar contingency table preview
both_correct   = ((df["base_correct"]) & (df["rep_correct"])).sum()
base_only      = ((df["base_correct"]) & (~df["rep_correct"])).sum()
repeat_only    = ((~df["base_correct"]) & (df["rep_correct"])).sum()
both_wrong     = ((~df["base_correct"]) & (~df["rep_correct"])).sum()

print("McNemar contingency table:")
print(f"  Both correct   : {both_correct}")
print(f"  Baseline only  : {base_only}   ← repeat hurt")
print(f"  Repeat only    : {repeat_only}  ← repeat helped")
print(f"  Both wrong     : {both_wrong}")
print()

# latency comparison (repeat is prefill-only overhead)
print(f"Median latency  baseline : {df['base_latency_ms'].median():.0f} ms")
print(f"Median latency  repeat   : {df['rep_latency_ms'].median():.0f} ms")
print()

# parse quality
print("Parse status (baseline):", df["base_parse_status"].value_counts().to_dict())
print("Parse status (repeat)  :", df["rep_parse_status"].value_counts().to_dict())

Questions     : 100
Baseline acc  : 69.0%  (69/100)
Repeat acc    : 66.0%  (66/100)
Δ accuracy    : -3.0%

McNemar contingency table:
  Both correct   : 64
  Baseline only  : 5   ← repeat hurt
  Repeat only    : 2  ← repeat helped
  Both wrong     : 29

Median latency  baseline : 2638 ms
Median latency  repeat   : 3456 ms

Parse status (baseline): {'exact_format': 100}
Parse status (repeat)  : {'exact_format': 100}


### 4/4/2026 Results after fixing arc_loader (1/2/3/4 -> A/B/C/D):
```
Questions     : 100
Baseline acc  : 69.0%  (69/100)
Repeat acc    : 66.0%  (66/100)
Δ accuracy    : -3.0%

McNemar contingency table:
  Both correct   : 64
  Baseline only  : 5   ← repeat hurt
  Repeat only    : 2  ← repeat helped
  Both wrong     : 29

Median latency  baseline : 2638 ms
Median latency  repeat   : 3456 ms

Parse status (baseline): {'exact_format': 100}
Parse status (repeat)  : {'exact_format': 100}
```

`NYSEDREGENTS_2008_8_27` now appears as a legitimate discordant result (repeat helped, gold=C), because the answer key is now correctly "C" instead of "3". However, it's also the one question still hitting the 256-token cap on the baseline side. 257 tokens is a very long chain-of-thought for a 7B model on a science MCQ, which is suspicious... Gonna treat it as a soft artifact even though the cap fix helped.

Overall accuracy nudged up slightly: baseline 67% to 69%, repeat 63% to 66%. Both gains attributable to the 3 previously poisoned questions now being evaluable; not actual progress. Doesn't matter in this case.

The headline result is unchanged and still not significant. The delta is -3% (was -4%), and the McNemar p-value is 0.45 on the full v2 set. Even excluding the capped discordant question gives p=0.22. There is no statistical evidence that prompt repetition hurts or helps Mistral 7B on ARC-Challenge at this sample size. 

`Mercury_411224` is still a concern. It was a cap artifact in v1 at 129 tokens. With max_tokens=256, it now generates 182 tokens (baseline) and 164 tokens (repeat). Repeat still flips from correct to wrong. This means it's now a genuine discordant result, not an artifact. The model is  reasoning itself into a different answer under the repeated prompt.

The verbosity pattern is holding. Wrong answers median 67 tokens (baseline) vs correct answers median 27 (roughly 2.5× longer). Very interesting. Might look into that.

Finding is a non-significant trend against repetition on this model at this scale, which is an honest and interesting divergence from the paper worth documenting.

### 4/2/2026 Results before fixing arc_loader (1/2/3/4 -/> A/B/C/D):
```
Questions     : 100
Baseline acc  : 67.0%  (67/100)
Repeat acc    : 63.0%  (63/100)
Δ accuracy    : -4.0%

McNemar contingency table:
  Both correct   : 62
  Baseline only  : 5   ← repeat hurt
  Repeat only    : 1  ← repeat helped
  Both wrong     : 32

Median latency  baseline : 2562 ms
Median latency  repeat   : 3388 ms

Parse status (baseline): {'exact_format': 100}
Parse status (repeat)  : {'exact_format': 100}
```

#### Thoughts:
McNemar's test statistic is roughly (5-1)² / (5+1) = 16/6 ≈ 2.67, which at 1 degree of freedom gives a p-value of about 0.10. 
* That's not significant at the conventional 0.05 threshold. 
* So the correct statement right now is: repeat prompt shows a non-significant trend toward hurting Mistral 7B on ARC-Challenge, not that it definitively hurts.

The paper found that prompt repetition helps non-reasoning models, but Mistral 7B 4-bit is a weaker and more heavily quantized model than anything tested in the original paper. A plausible explanation is that the doubled context confuses a smaller model rather than helping it attend to the question. The attention asymmetry benefit may require sufficient model capacity to realize.

Parse quality is still perfect, which means this result is clean (not a measurement artifact).

*see analysis/poc_qualitative_audit.ipynb to see what questions where wrong and why*

##### Individual Results

In [8]:
display(df[[
    "question_id", "gold",
    "base_predicted", "base_correct",
    "rep_predicted",  "rep_correct",
    "base_parse_status", "rep_parse_status",
]].style.apply(
    lambda col: ["background: #d4edda" if v else "background: #f8d7da" for v in col]
    if col.name in ("base_correct", "rep_correct") else [""] * len(col),
    axis=0
))

,question_id,gold,base_predicted,base_correct,rep_predicted,rep_correct,base_parse_status,rep_parse_status
0,Mercury_SC_407400,C,D,False,D,False,exact_format,exact_format
1,MCAS_2003_5_33,C,C,True,C,True,exact_format,exact_format
2,Mercury_7235638,C,B,False,B,False,exact_format,exact_format
3,Mercury_SC_416529,B,B,True,B,True,exact_format,exact_format
4,Mercury_406777,A,A,True,A,True,exact_format,exact_format
5,MCAS_2006_9_43,C,A,False,A,False,exact_format,exact_format
6,MCAS_2014_8_20,A,A,True,A,True,exact_format,exact_format
7,Mercury_7211680,C,C,True,C,True,exact_format,exact_format
8,Mercury_7064575,B,B,True,B,True,exact_format,exact_format
9,NCEOGA_2013_5_20,B,B,True,B,True,exact_format,exact_format


#### Save results to disk

In [9]:
import pathlib

results_dir = pathlib.Path("../experiments/results")
results_dir.mkdir(parents=True, exist_ok=True)

out_path = results_dir / "poc_arc_mlx_mistral7b_100_v2.json"
df.to_json(out_path, orient="records", indent=2)
print(f"Results saved to: {out_path}")

Results saved to: ../experiments/results/poc_arc_mlx_mistral7b_100_v2.json


## Unload Model

In [10]:
# Do this before switching to API-based agents or closing the notebook.
# The ~4.5 GB will be returned to the system memory pool.

agent.unload()
print("Model unloaded. Unified memory is free for other processes.")

INFO  MLXAgent: model unloaded.


Model unloaded. Unified memory is free for other processes.
